In [ ]:
!pip install simpy

In [ ]:
import simpy
import random
import statistics

In [ ]:
def cliente(env, name, caja, cocina, results):
    t_llegada = env.now

In [ ]:
def cliente(env, name, caja, cocina, results):
    t_llegada = env.now

    # 1. Proceso en Caja (Recurso compartido de 2 cajeros)
    with caja.request() as req_caja:
        yield req_caja  # Espera a que se desocupe un cajero
        t_espera_caja = env.now - t_llegada

        # Tiempo en caja: Dist. triangular (mín=3, moda=5, máx=10 minutos por receso)
        t_caja = random.triangular(3, 5, 10)
        yield env.timeout(t_caja)

    t_fin_caja = env.now

    # 2. Proceso en Cocina (Recurso compartido de 2 cocineros)
    with cocina.request() as req_cocina:
        yield req_cocina  # Espera a que se desocupe un cocinero
        t_espera_cocina = env.now - t_fin_caja

        # Tiempo en cocina: Dist. triangular (mín=10, moda=15, máx=30 minutos por receso)
        t_cocina = random.triangular(10, 15, 30)
        yield env.timeout(t_cocina)

    t_salida = env.now
    t_total = t_salida - t_llegada

    # Guardamos los registros de tiempos de este cliente
    results.append({
        'espera_caja': t_espera_caja,
        'espera_cocina': t_espera_cocina,
        'tiempo_total': t_total
    })

In [ ]:
def generador_clientes(env, caja, cocina, results):
    i = 0
    while True:
        # Tiempo entre llegadas con distribución exponencial (promedio 1.5 min)
        yield env.timeout(random.expovariate(1.0 / 1.5))
        i += 1
        env.process(cliente(env, f'Cliente_{i}', caja, cocina, results))

In [ ]:
resultados_totales = []
SEMILLA_BASE = 42
NUM_CORRIDAS = 30

for r in range(NUM_CORRIDAS):
    # Asignar una semilla diferente en cada corrida para asegurar variabilidad estocástica
    random.seed(SEMILLA_BASE + r)

    env = simpy.Environment()

    # Definir los recursos con la capacidad real de la cafetería
    caja = simpy.Resource(env, capacity=2)    # 2 cajeros
    cocina = simpy.Resource(env, capacity=2)  # 2 cocineros

    resultados_corrida = []
    env.process(generador_clientes(env, caja, cocina, resultados_corrida))

    # Correr cada simulación durante 120 minutos de operación
    env.run(until=120)

    resultados_totales.extend(resultados_corrida)

# Calcular estadísticas generales de las 30 corridas
prom_espera_caja = statistics.mean([r['espera_caja'] for r in resultados_totales])
prom_espera_cocina = statistics.mean([r['espera_cocina'] for r in resultados_totales])
prom_tiempo_total = statistics.mean([r['tiempo_total'] for r in resultados_totales])

print(f"--- RESULTADOS FINALES DE LAS {NUM_CORRIDAS} CORRIDAS ---")
print(f"Clientes totales simulados: {len(resultados_totales)}")
print(f"Promedio de espera en caja: {prom_espera_caja:.2f} minutos")
print(f"Promedio de espera en cocina: {prom_espera_cocina:.2f} minutos")
print(f"Promedio de tiempo total en el sistema: {prom_tiempo_total:.2f} minutos")

--- RESULTADOS FINALES DE LAS 30 CORRIDAS ---
Clientes totales simulados: 375
Promedio de espera en caja: 6.66 minutos
Promedio de espera en cocina: 29.09 minutos
Promedio de tiempo total en el sistema: 57.80 minutos
